In [1]:
## IMPORT REQUIRED LIBRARIES

import pandas as pd
import duckdb

print("Libraries Loaded")

Libraries Loaded


In [2]:
## LOAD THE DATASET

df = pd.read_csv("../data/processed/hotel_bookings_cleaned.csv")

print(df.shape)

df.head()

(119390, 36)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_nights,total_guests,arrival_date_num,stay_type
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,0,2.0,7,Short Stay
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,Check-Out,2015-07-01,0,2.0,7,Short Stay
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,1,1.0,7,Short Stay
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,Check-Out,2015-07-02,1,1.0,7,Short Stay
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,Check-Out,2015-07-03,2,2.0,7,Short Stay


In [3]:
## REGISTER DATABASE INTO DUCKDB

duckdb.register(
    "hotel_data",
    df
)

print("Table Registered")

Table Registered


In [ ]:
## BOOKING COUNT BY HOTEL type

duckdb.sql("""      
SELECT 
    hotel, 
    COUNT(*) AS total_bookings
FROM hotel_data
GROUP BY hotel
ORDER BY total_bookings DESC          
""").df()


,hotel,total_bookings
0,City Hotel,79330
1,Resort Hotel,40060


### Observation
- City Hotels account for approximately 66% of total bookings.
- Resort Hotels account for approximately 34% of total bookings.
- City Hotels are significantly more popular than Resort Hotels.

In [ ]:
## CANCELLATION RATE BY HOTEL

duckdb.sql("""
SELECT 
    hotel, 
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY hotel
ORDER BY cancellation_rate DESC
""").df()


,hotel,total_bookings,cancelled_bookings,cancellation_rate
0,City Hotel,79330,33102.0,41.73
1,Resort Hotel,40060,11122.0,27.76


### Observation
- City Hotels experience substantially higher cancellations.
- Nearly 42% of City Hotel bookings are cancelled.
- Resort Hotels have better booking retention.
- Hotel type appears to be an important predictor of cancellation behavior.

In [ ]:
## OVERALL CANCELLATION RATE

duckdb.sql("""
SELECT 
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS overall_cancellation_rate
FROM hotel_data
""").df()


,total_bookings,cancelled_bookings,overall_cancellation_rate
0,119390,44224.0,37.04


### Observation
- More than one-third of all hotel bookings are cancelled.
- Cancellation management can significantly improve revenue stability.
- Building a cancellation prediction model is justified.

In [ ]:
## ADR BY HOTEL TYPE

duckdb.sql("""
SELECT
    hotel,
    ROUND(AVG(adr), 2) AS avg_adr,
FROM hotel_data
GROUP BY hotel
ORDER BY avg_adr DESC
""").df()


,hotel,avg_adr
0,City Hotel,105.30
1,Resort Hotel,94.95


### Observation

- City Hotels charge higher average daily rates.
- Despite higher ADR, City Hotels also suffer higher cancellations.
- Revenue optimization and cancellation management should be jointly analyzed.

In [ ]:
## CANCELLATION BY MARKET SEGMENT

duckdb.sql("""
SELECT 
    market_segment,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY market_segment
ORDER BY cancellation_rate DESC
""").df()


,market_segment,total_bookings,cancelled_bookings,cancellation_rate
0,Undefined,2,2.0,100.00
1,Groups,19811,12097.0,61.06
2,Online TA,56477,20739.0,36.72
3,Offline TA/TO,24219,8311.0,34.32
4,Aviation,237,52.0,21.94
5,Corporate,5295,992.0,18.73
6,Direct,12606,1934.0,15.34
7,Complementary,743,97.0,13.06


### Observation

- Group bookings exhibit the highest cancellation rate.
- Online Travel Agencies contribute a large volume of cancellations.
- Direct bookings are considerably more reliable.
- Marketing strategies should encourage direct bookings.

In [9]:
## CUSTOMER TYPE ANALYSIS

duckdb.sql("""
SELECT
    customer_type,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY customer_type
ORDER BY cancellation_rate DESC
""").df()


,customer_type,total_bookings,cancelled_bookings,cancellation_rate
0,Transient,89613,36514.0,40.75
1,Contract,4076,1262.0,30.96
2,Transient-Party,25124,6389.0,25.43
3,Group,577,59.0,10.23


In [13]:
## DEPOSIT TYPE ANALYSIS

duckdb.sql("""
SELECT
    deposit_type,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS cancelled_bookings,
    ROUND(AVG(is_canceled)*100, 2) AS cancellation_rate
FROM hotel_data
GROUP BY deposit_type
ORDER BY cancellation_rate DESC
""").df()


,deposit_type,total_bookings,cancelled_bookings,cancellation_rate
0,Non Refund,14587,14494.0,99.36
1,No Deposit,104641,29694.0,28.38
2,Refundable,162,36.0,22.22


In [14]:
## AVERAGE LEAD TIME BY CANCELLATION

duckdb.sql("""
SELECT
    is_canceled,
    ROUND(AVG(lead_time) *100, 2) AS avg_lead_time
FROM hotel_data
GROUP BY is_canceled
""").df()

,is_canceled,avg_lead_time
0,0,7998.47
1,1,14484.88
